# Salud Chilecito - Demo DAO y plataforma

Este notebook funciona como recorrido de presentacion de Salud Chilecito. Primero se valida el modelo de datos de demo, despues se revisa el DAO y finalmente se explican los comandos para conectar con Oracle.

**Formas de uso del proyecto:**

- `http://localhost:8000`: plataforma grafica.
- `http://localhost:8000/bot`: plataforma conversacional con Bot IA local.
- `src/dao`: capa DAO para Oracle.
- `sql/`: scripts Oracle con tablespaces, usuarios, esquema, indices, seed y validaciones.


## 1. Preparacion

Ejecutar el notebook desde la raiz del repositorio:

```bash
Windows: .\scripts\windows\04_abrir_notebook.ps1
Ubuntu: bash scripts/ubuntu/04_abrir_notebook.sh
```

Si todavia no se instalaron dependencias:

```bash
python -m venv .venv
.venv\Scripts\activate        # Windows
source .venv/bin/activate      # Ubuntu
pip install -r requirements.txt
```


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print("Repositorio:", ROOT)
print("Existe src/dao:", (ROOT / "src" / "dao").exists())
print("Existe sql/:", (ROOT / "sql").exists())


## 2. Datos de demo

La plataforma web puede funcionar aunque Oracle no este levantado. Para eso usa `data/demo_seed.json` y guarda cambios de prueba en `runtime/salud_chilecito_data.json`.


In [ ]:
seed_path = ROOT / "data" / "demo_seed.json"
data = json.loads(seed_path.read_text(encoding="utf-8"))

for name in ["centros", "especialidades", "medicos", "pacientes", "turnos", "agendas", "tarifas", "documentos"]:
    print(f"{name}: {len(data[name])}")


In [ ]:
import pandas as pd

pd.DataFrame(data["centros"])


## 3. Uso del store local

Esta parte demuestra operaciones reales sin tocar Oracle. Sirve para probar la interfaz y validar la funcionalidad sin preparar una base externa.


In [ ]:
from tempfile import TemporaryDirectory
from src.webapp.store import JsonStore

tmp = TemporaryDirectory()
demo_store = JsonStore(
    runtime_path=Path(tmp.name) / "runtime.json",
    seed_path=seed_path,
    uploads_dir=Path(tmp.name) / "uploads",
)

paciente = demo_store.create_patient({
    "dni": "50999888",
    "nombre": "Paciente Notebook",
    "telefono": "3825-111111",
    "distrito": "Chilecito",
    "obra_social": "APOS",
})

turno = demo_store.create_turno({
    "paciente_id": paciente["id"],
    "medico_id": 1,
    "fecha": "2026-06-20",
    "hora": "09:30",
    "estado": "CONFIRMADO",
    "precio": 0,
    "motivo": "Control desde notebook",
})

print("Paciente creado:", paciente)
print("Turno creado:", turno["id"], turno["paciente"]["nombre"], turno["medico"]["nombre"])


## 4. Bot IA local

El bot no llama a servicios externos: interpreta comandos frecuentes y opera el mismo store de la plataforma. Esto permite demostrar una segunda forma de uso, conversacional, sin depender de una API paga.


In [ ]:
from src.webapp.bot_agent import BotAgent

bot = BotAgent(demo_store)
respuesta = bot.handle("listar turnos")
print(respuesta["reply"])


## 5. Capa DAO para Oracle

Los DAO son la interfaz formal entre Python y Oracle. La aplicacion no deberia armar consultas dispersas por todos lados; las consultas quedan agrupadas en clases como `CentroDAO`, `PacienteDAO` y `TurnoDAO`.


In [ ]:
from src.dao import CentroDAO, PacienteDAO, TurnoDAO

print("DAO disponibles:")
print("-", CentroDAO.__name__)
print("-", PacienteDAO.__name__)
print("-", TurnoDAO.__name__)


In [ ]:
## 5. Capa DAO para Oracle

Los DAO son la interfaz formal entre Python y Oracle. La aplicacion no deberia armar consultas dispersas por todos lados; las consultas quedan agrupadas en clases como `CentroDAO`, `PacienteDAO` y `TurnoDAO.

**Nuevos métodos del modelo Single-Hospital:**
- `listar_sintomas()` - Lista síntomas con especialidades
- `buscar_especialidad_por_sintoma(sintoma)` - Busca especialidad por síntoma
- `obtener_configuracion_hospital(id)` - Configuración del hospital
- `listar_tipos_consulta()` - Tipos de consulta disponibles
- `obtener_precios_por_especialidad(centro, especialidad)` - Precios por especialidad

In [ ]:
## 5.1 Documentación Completa del DAO - Estructura de Tablas y Operaciones

### Estructura de Tablas de la Base de Datos

El sistema utiliza las siguientes tablas en Oracle:

#### 1. CENTROS
```sql
CREATE TABLE centros (
    id NUMBER PRIMARY KEY,
    nombre VARCHAR2(100) NOT NULL,
    direccion VARCHAR2(200),
    distrito VARCHAR2(100),
    telefono VARCHAR2(50),
    tipo VARCHAR2(20) CHECK (tipo IN ('HOSPITAL', 'CLINICA', 'CENTRO_SALUD'))
);
```

#### 2. ESPECIALIDADES
```sql
CREATE TABLE especialidades (
    id NUMBER PRIMARY KEY,
    nombre VARCHAR2(100) NOT NULL,
    descripcion VARCHAR2(500)
);
```

#### 3. MEDICOS
```sql
CREATE TABLE medicos (
    id NUMBER PRIMARY KEY,
    nombre VARCHAR2(100) NOT NULL,
    especialidad_id NUMBER REFERENCES especialidades(id),
    centro_id NUMBER REFERENCES centros(id),
    telefono VARCHAR2(50)
);
```

#### 4. PACIENTES
```sql
CREATE TABLE pacientes (
    id NUMBER PRIMARY KEY,
    dni VARCHAR2(20) UNIQUE NOT NULL,
    nombre VARCHAR2(100) NOT NULL,
    telefono VARCHAR2(50),
    obra_social VARCHAR2(100),
    distrito VARCHAR2(100),
    centro_id NUMBER REFERENCES centros(id)  -- Para asociación multi-hospital
);
```

#### 5. TURNOS
```sql
CREATE TABLE turnos (
    id NUMBER PRIMARY KEY,
    paciente_id NUMBER REFERENCES pacientes(id),
    medico_id NUMBER REFERENCES medicos(id),
    centro_id NUMBER REFERENCES centros(id),
    fecha DATE NOT NULL,
    hora VARCHAR2(5) NOT NULL,
    estado VARCHAR2(20) CHECK (estado IN ('PENDIENTE', 'CONFIRMADO', 'ATENDIDO', 'CANCELADO', 'AUSENTE')),
    precio NUMBER,
    motivo VARCHAR2(500),
    origen VARCHAR2(20) DEFAULT 'VIRTUAL',  -- VIRTUAL o FISICO
    fecha_creacion TIMESTAMP DEFAULT SYSTIMESTAMP
);
```

#### 6. AGENDAS
```sql
CREATE TABLE agendas (
    id NUMBER PRIMARY KEY,
    medico_id NUMBER REFERENCES medicos(id),
    dia_semana VARCHAR2(20) NOT NULL,
    hora_inicio VARCHAR2(5) NOT NULL,
    hora_fin VARCHAR2(5) NOT NULL,
    duracion_minutos NUMBER NOT NULL,
    cupo_diario NUMBER NOT NULL
);
```

#### 7. DOCUMENTOS
```sql
CREATE TABLE documentos (
    id NUMBER PRIMARY KEY,
    paciente_id NUMBER REFERENCES pacientes(id),
    tipo VARCHAR2(50),
    nombre_archivo VARCHAR2(200),
    ruta VARCHAR2(500),
    mime_type VARCHAR2(100),
    tamano_bytes NUMBER
);
```

#### 8. SINTOMAS (Modelo Single-Hospital)
```sql
CREATE TABLE sintomas (
    id NUMBER PRIMARY KEY,
    descripcion VARCHAR2(200) NOT NULL,
    especialidad_id NUMBER REFERENCES especialidades(id),
    prioridad VARCHAR2(20) CHECK (prioridad IN ('ALTA', 'MEDIA', 'BAJA'))
);
```

#### 9. CONFIGURACION_HOSPITAL (Modelo Single-Hospital)
```sql
CREATE TABLE configuracion_hospital (
    id NUMBER PRIMARY KEY,
    nombre_hospital VARCHAR2(100),
    centro_principal NUMBER REFERENCES centros(id),
    mensaje_bienvenida VARCHAR2(500),
    color_primario VARCHAR2(20),
    color_secundario VARCHAR2(20),
    tiempo_cancelacion_horas NUMBER DEFAULT 24
);
```

#### 10. TIPOS_CONSULTA (Modelo Single-Hospital)
```sql
CREATE TABLE tipos_consulta (
    id NUMBER PRIMARY KEY,
    nombre VARCHAR2(50) NOT NULL,
    descripcion VARCHAR2(200)
);
```

#### 11. PRECIOS_ESPECIALIDAD (Modelo Single-Hospital)
```sql
CREATE TABLE precios_especialidad (
    id NUMBER PRIMARY KEY,
    centro_id NUMBER REFERENCES centros(id),
    especialidad_id NUMBER REFERENCES especialidades(id),
    tipo_consulta_id NUMBER REFERENCES tipos_consulta(id),
    precio_estimado NUMBER,
    precio_minimo NUMBER,
    precio_maximo NUMBER,
    duracion_minutos NUMBER
);
```

---

### Operaciones del DAO - Ejemplos Completos

#### Clase SaludDAO

```python
from dao import SaludDAO
from cx_Oracle import makedsn

# Conexión a Oracle
dsn = makedsn("localhost", 1521, "XE")
connection = cx_Oracle.connect("user", "password", dsn)
dao = SaludDAO(connection)
```

#### 1. Operaciones de Conexión

```python
# Verificar conexión
dao.ping()  # Retorna "Conectado a Oracle"

# Contar registros en una tabla
total_centros = dao.contar("centros")
total_pacientes = dao.contar("pacientes")
print(f"Centros: {total_centros}, Pacientes: {total_pacientes}")
```

#### 2. Operaciones CRUD - Centros

```python
# Listar todos los centros
centros = dao.listar_centros()
for centro in centros:
    print(f"{centro['nombre']} - {centro['distrito']}")

# Crear un nuevo centro
nuevo_centro = dao.crear_centro(
    nombre="Hospital San Martín",
    direccion="Av. Principal 123",
    distrito="Chilecito",
    telefono="3825-555555",
    tipo="HOSPITAL"
)
print(f"Centro creado con ID: {nuevo_centro['id']}")

# Buscar centro por nombre
centro = dao.obtener_centro_por_nombre("Hospital San Martín")
print(f"Centro encontrado: {centro}")

# Actualizar centro
dao.actualizar_centro(
    centro_id=nuevo_centro['id'],
    nombre="Hospital San Martín Actualizado",
    direccion="Av. Principal 456",
    distrito="Chilecito",
    telefono="3825-666666",
    tipo="HOSPITAL"
)
```

#### 3. Operaciones CRUD - Especialidades

```python
# Listar especialidades
especialidades = dao.listar_especialidades()
for esp in especialidades:
    print(f"{esp['nombre']}")

# Crear especialidad
nueva_especialidad = dao.crear_especialidad(
    nombre="Traumatología",
    description="Especialidad médica para tratamiento de lesiones"
)
print(f"Especialidad creada con ID: {nueva_especialidad['id']}")

# Buscar especialidad por nombre
especialidad = dao.obtener_especialidad_por_nombre("Cardiología")
print(f"Especialidad encontrada: {especialidad}")
```

#### 4. Operaciones CRUD - Médicos

```python
# Listar médicos por centro
medicos = dao.listar_medicos_por_centro(centro_id=1)
for medico in medicos:
    print(f"Dr. {medico['nombre']} - {medico['especialidad']}")

# Buscar médicos por especialidad
medicos_cardiologia = dao.buscar_medicos_por_especialidad(especialidad_id=1)
for medico in medicos_cardiologia:
    print(f"Dr. {medico['nombre']}")

# Crear médico
nuevo_medico = dao.crear_medico(
    nombre="Dr. Juan Pérez",
    especialidad_id=1,
    centro_id=1,
    telefono="3825-777777"
)
print(f"Médico creado con ID: {nuevo_medico['id']}")
```

#### 5. Operaciones CRUD - Pacientes

```python
# Listar pacientes
pacientes = dao.listar_pacientes()
for paciente in pacientes:
    print(f"{paciente['nombre']} - DNI: {paciente['dni']}")

# Buscar paciente por DNI
paciente = dao.obtener_paciente_por_dni("12345678")
print(f"Paciente encontrado: {paciente}")

# Crear paciente (con centro_id para multi-hospital)
nuevo_paciente = dao.crear_paciente(
    dni="87654321",
    nombre="María García",
    telefono="3825-888888",
    obra_social="APOS",
    distrito="Chilecito",
    centro_id=1  # Asociado al hospital
)
print(f"Paciente creado con ID: {nuevo_paciente['id']}")

# Actualizar contacto de paciente
dao.actualizar_contacto_paciente(
    paciente_id=nuevo_paciente['id'],
    telefono="3825-999999",
    obra_social="OSDE"
)
```

#### 6. Operaciones CRUD - Turnos

```python
# Listar turnos próximos
turnos = dao.listar_turnos_proximos(dias=7)
for turno in turnos:
    print(f"{turno['fecha']} {turno['hora']} - {turno['paciente']['nombre']}")

# Reservar turno (con validación de disponibilidad en tiempo real)
turno = dao.reservar_turno(
    paciente_id=1,
    medico_id=1,
    fecha="2026-06-20",
    hora="09:30",
    precio=5000,
    motivo="Consulta de rutina"
)
print(f"Turno reservado con ID: {turno['id']}")

# Cambiar estado de turno
dao.cambiar_estado_turno(turno_id=turno['id'], estado="CONFIRMADO")

# Eliminar turno
dao.eliminar_turno(turno_id=turno['id'])

# Disponibilidad por médico
disponibilidad = dao.disponibilidad_por_medico(medico_id=1)
for slot in disponibilidad:
    print(f"{slot['fecha']} - Cupos: {slot['cupos_disponibles']}")
```

#### 7. Operaciones CRUD - Agendas

```python
# Listar agenda de un médico
agenda = dao.listar_agenda_por_medico(medico_id=1)
for slot in agenda:
    print(f"{slot['dia_semana']} {slot['hora_inicio']}-{slot['hora_fin']}")

# Crear agenda
nueva_agenda = dao.crear_agenda(
    medico_id=1,
    dia_semana="Lunes",
    hora_inicio="08:00",
    hora_fin="12:00",
    duracion_minutos=30,
    cupo_diario=8
)
print(f"Agenda creada con ID: {nueva_agenda['id']}")
```

#### 8. Operaciones CRUD - Documentos

```python
# Listar documentos de un paciente
documentos = dao.listar_documentos_por_paciente(paciente_id=1)
for doc in documentos:
    print(f"{doc['nombre_archivo']} - {doc['tipo']}")

# Crear documento
nuevo_documento = dao.crear_documento(
    paciente_id=1,
    tipo="ANALISIS",
    nombre_archivo="analisis_sangre.pdf",
    ruta="/uploads/paciente_1_1_analisis_sangre.pdf",
    mime_type="application/pdf",
    tamano_bytes=1024000
)
print(f"Documento creado con ID: {nuevo_documento['id']}")
```

#### 9. Operaciones - Historial Clínico

```python
# Listar historial de un paciente
historial = dao.listar_historial_por_paciente(paciente_id=1)
for registro in historial:
    print(f"{registro['fecha']} - {registro['motivo']}")

# Registrar en historial
dao.registrar_historial(
    paciente_id=1,
    medico_id=1,
    fecha="2026-06-20",
    motivo="Control de presión arterial",
    diagnostico="Hipertensión leve",
    tratamiento="Reposo y medicación"
)
```

#### 10. Operaciones - Síntomas (Modelo Single-Hospital)

```python
# Listar síntomas con especialidades
sintomas = dao.listar_sintomas()
for s in sintomas:
    print(f"{s['descripcion']} → {s['especialidad']} (Prioridad: {s['prioridad']})")

# Buscar especialidad por síntoma
resultado = dao.buscar_especialidad_por_sintoma("dolor de pecho")
if resultado:
    print(f"Para 'dolor de pecho' se recomienda: {resultado['especialidad']}")
    print(f"Prioridad: {resultado['prioridad']}")

# Crear síntoma
nuevo_sintoma = dao.crear_sintoma(
    descripcion="Dolor abdominal",
    especialidad_id=2,  # Gastroenterología
    prioridad="MEDIA"
)
print(f"Síntoma creado con ID: {nuevo_sintoma['id']}")
```

#### 11. Operaciones - Configuración del Hospital (Modelo Single-Hospital)

```python
# Obtener configuración del hospital
config = dao.obtener_configuracion_hospital(id=1)
if config:
    print(f"Nombre: {config['nombre_hospital']}")
    print(f"Mensaje: {config['mensaje_bienvenida']}")
    print(f"Colores: {config['color_primario']}, {config['color_secundario']}")

# Crear configuración del hospital
nueva_config = dao.crear_configuracion_hospital(
    nombre_hospital="Hospital San Martín",
    centro_principal=1,
    mensaje_bienvenida="Bienvenido al Hospital San Martín",
    color_primario="#0066cc",
    color_secundario="#ffffff",
    tiempo_cancelacion_horas=24
)
print(f"Configuración creada con ID: {nueva_config['id']}")
```

#### 12. Operaciones - Tipos de Consulta y Precios (Modelo Single-Hospital)

```python
# Listar tipos de consulta
tipos = dao.listar_tipos_consulta()
for tipo in tipos:
    print(f"{tipo['nombre']} - {tipo['descripcion']}")

# Obtener precios por especialidad
precios = dao.obtener_precios_por_especialidad(centro_id=1, especialidad_id=1)
for precio in precios:
    print(f"{precio['tipo_consulta']['nombre']}: ${precio['precio_estimado']}")
    print(f"  Rango: ${precio['precio_minimo']} - ${precio['precio_maximo']}")
    print(f"  Duración: {precio['duracion_minutos']} minutos")

# Obtener precio estimado por tipo
precio = dao.obtener_precio_estimado_por_tipo(
    centro_id=1,
    especialidad_id=1,
    tipo_consulta_id=1
)
print(f"Precio estimado: ${precio}")
```

#### 13. Operaciones - Disponibilidad Mejorada (Modelo Single-Hospital)

```python
# Obtener turnos disponibles por médico (próximos 7 días)
disponibilidad = dao.obtener_turnos_disponibles_por_medico(medico_id=1, dias=7)
for slot in disponibilidad:
    print(f"Fecha: {slot['fecha']}")
    print(f"Día: {slot['dia_semana']}")
    print(f"Horario: {slot['hora_inicio']} - {slot['hora_fin']}")
    print(f"Cupos: {slot['cupos_disponibles']}/{slot['cupo_diario']}")
    print(f"Duración: {slot['duracion_minutos']} minutos")
    print()

# Obtener médicos disponibles por especialidad
medicos = dao.obtener_medicos_disponibles_por_especialidad(
    especialidad_id=1,
    fecha="2026-06-20"
)
for medico in medicos:
    print(f"Dr. {medico['nombre']} - {medico['especialidad']}")

# Obtener horarios disponibles para una fecha específica
horarios = dao.obtener_horarios_disponibles(medico_id=1, fecha="2026-06-20")
for h in horarios:
    print(f"{h['hora']} - {h['hora_fin']}")
```

---

### Funcionamiento Completo del Proyecto

#### Flujo de Datos

1. **Frontend (JavaScript)** → API REST → Backend (Python) → DAO → Oracle
2. **Bot IA** → Store → Backend → DAO → Oracle
3. **Integración HIS** → API Keys → Webhooks → Adaptadores → HIS Externo

#### Arquitectura de Capas

```
┌─────────────────────────────────────────────────────────┐
│                    Frontend (JS)                        │
│  - Plataforma gráfica (index.html, app.js)             │
│  - Bot conversacional (bot.html, bot.js)                │
│  - Landing page (landing.html, landing.js)              │
└────────────────────┬────────────────────────────────────┘
                     │ API REST
┌────────────────────▼────────────────────────────────────┐
│                  Backend (Python)                        │
│  - Servidor HTTP (server.py)                            │
│  - Store JSON (store.py)                                │
│  - Bot Agent (bot_agent.py)                             │
│  - Autenticación (auth.py)                              │
│  - Webhooks (webhooks.py)                               │
│  - Adaptadores (adapters.py)                            │
└────────────────────┬────────────────────────────────────┘
                     │
┌────────────────────▼────────────────────────────────────┐
│                    Capa DAO (dao.py)                      │
│  - SaludDAO (clase principal)                           │
│  - CentroDAO, PacienteDAO, TurnoDAO, etc.              │
│  - Abstracción de base de datos                         │
└────────────────────┬────────────────────────────────────┘
                     │ Oracle SQL
┌────────────────────▼────────────────────────────────────┐
│              Base de Datos Oracle                        │
│  - Tablas: centros, especialidades, médicos, etc.      │
│  - Indexes, Constraints, Triggers                       │
└─────────────────────────────────────────────────────────┘
```

#### Ciclo de Vida de un Turno

1. **Paciente selecciona síntomas** → Sistema sugiere especialidad
2. **Paciente elige médico y horario** → Sistema verifica disponibilidad
3. **Sistema valida disponibilidad en tiempo real** → Bloquea cupo
4. **Turno se crea con estado PENDIENTE** → Se guarda en Oracle
5. **Turno se confirma** → Estado cambia a CONFIRMADO
6. **Paciente asiste** → Estado cambia a ATENDIDO
7. **Paciente no asiste** → Estado cambia a AUSENTE
8. **Turno se cancela** → Estado cambia a CANCELADO → Cupo se libera

#### Sincronización Virtual/Físico

- **Virtual**: Reserva desde plataforma web (origen: "VIRTUAL")
- **Físico**: Reserva presencial en hospital (origen: "FISICO")
- **Validación**: Ambos canales verifican disponibilidad antes de reservar
- **Prevención**: Sistema previene double-booking en tiempo real

#### Persistencia de Datos

- **Guardado automático**: Cada operación guarda en disco inmediatamente
- **Logging**: Sistema registra cada operación de guardado
- **Verificación**: Frontend verifica estado de persistencia al cargar
- **Guardado manual**: Botón "Guardar" para forzar guardado manual
- **Fallback**: Si hay error de escritura, usa memoria como fallback

#### Integración con HIS

- **API Keys**: Autenticación segura para integraciones
- **Webhooks**: Sincronización bidireccional en tiempo real
- **Adaptadores**: REST y FHIR para diferentes tipos de HIS
- **Logs de auditoría**: Registro de todas las operaciones de integración

---

### Scripts SQL Disponibles

En el directorio `sql/` encontrarás:

- `01_tablespaces.sql` - Creación de tablespaces
- `02_usuario.sql` - Creación de usuario y permisos
- `03_esquema.sql` - Creación de tablas y estructura
- `04_indices.sql` - Creación de índices para rendimiento
- `05_seed.sql` - Datos iniciales de demostración
- `06_validaciones.sql` - Constraints y triggers

Para ejecutarlos:

```bash
sqlplus system/password @sql/01_tablespaces.sql
sqlplus system/password @sql/02_usuario.sql
sqlplus salud_chilecito/password @sql/03_esquema.sql
sqlplus salud_chilecito/password @sql/04_indices.sql
sqlplus salud_chilecito/password @sql/05_seed.sql
sqlplus salud_chilecito/password @sql/06_validaciones.sql
```

---

### Resumen de Operaciones del DAO

| Operación | Método | Descripción |
|-----------|--------|-------------|
| **Centros** | listar_centros() | Lista todos los centros |
| | crear_centro() | Crea un nuevo centro |
| | obtener_centro_por_nombre() | Busca centro por nombre |
| | actualizar_centro() | Actualiza datos de centro |
| **Especialidades** | listar_especialidades() | Lista todas las especialidades |
| | crear_especialidad() | Crea una especialidad |
| | obtener_especialidad_por_nombre() | Busca especialidad por nombre |
| **Médicos** | listar_medicos_por_centro() | Lista médicos de un centro |
| | buscar_medicos_por_especialidad() | Busca médicos por especialidad |
| | crear_medico() | Crea un médico |
| **Pacientes** | listar_pacientes() | Lista todos los pacientes |
| | obtener_paciente_por_dni() | Busca paciente por DNI |
| | crear_paciente() | Crea un paciente (con centro_id) |
| | actualizar_contacto_paciente() | Actualiza contacto de paciente |
| **Turnos** | listar_turnos_proximos() | Lista turnos próximos |
| | reservar_turno() | Reserva un turno (con validación) |
| | cambiar_estado_turno() | Cambia estado de turno |
| | eliminar_turno() | Elimina un turno |
| | disponibilidad_por_medico() | Consulta disponibilidad |
| **Agendas** | listar_agenda_por_medico() | Lista agenda de médico |
| | crear_agenda() | Crea agenda |
| **Documentos** | listar_documentos_por_paciente() | Lista documentos de paciente |
| | crear_documento() | Crea documento |
| **Historial** | listar_historial_por_paciente() | Lista historial de paciente |
| | registrar_historial() | Registra en historial |
| **Síntomas** | listar_sintomas() | Lista síntomas con especialidades |
| | buscar_especialidad_por_sintoma() | Deriva por síntoma |
| | crear_sintoma() | Crea síntoma |
| **Configuración** | obtener_configuracion_hospital() | Obtiene config del hospital |
| | crear_configuracion_hospital() | Crea configuración |
| **Precios** | listar_tipos_consulta() | Lista tipos de consulta |
| | obtener_precios_por_especialidad() | Precios por especialidad |
| | obtener_precio_estimado_por_tipo() | Precio específico |
| **Disponibilidad** | obtener_turnos_disponibles_por_medico() | Turnos disponibles |
| | obtener_medicos_disponibles_por_especialidad() | Médicos disponibles |
| | obtener_horarios_disponibles() | Horarios específicos |

class FakeDB:
    def __init__(self):
        self.calls = []

    def fetch_all(self, sql, params=None):
        self.calls.append(("fetch_all", " ".join(sql.split()), params or {}))
        return []

    def fetch_one(self, sql, params=None):
        self.calls.append(("fetch_one", " ".join(sql.split()), params or {}))
        return {"total": 0}

    def execute(self, sql, params=None):
        self.calls.append(("execute", " ".join(sql.split()), params or {}))
        return 1

fake_db = FakeDB()
CentroDAO(fake_db).listar(distrito="Chilecito")
fake_db.calls[-1]

## 6. Sistema de Integración con HIS (Hospital Information Systems)

**Problema**: Los hospitales ya tienen sistemas de gestión (HIS) y no quieren reemplazarlos.

**Solución**: Salud Chilecito se integra con los sistemas existentes mediante API REST, no los reemplaza.

### Características del sistema de integración

- **API Keys**: Autenticación segura para integraciones entre sistemas
- **Webhooks**: Sincronización bidireccional en tiempo real
- **Adaptadores**: Soporte para diferentes tipos de HIS (REST, FHIR)
- **Logs de auditoría**: Registro de todas las operaciones de integración

### Ejemplo de uso de API Keys

```python
from src.webapp.auth import api_key_manager

# Generar una API Key para un hospital
api_key = api_key_manager.generate_key(
    hospital_name="Hospital Eleazar Herrera Motta",
    hospital_id=1,
    permissions=["read", "write"]
)
print(f"API Key generada: {api_key}")

# Validar la API Key
hospital_info = api_key_manager.validate_key(api_key)
print(f"Información del hospital: {hospital_info}")
```

### Ejemplo de uso de Webhooks

```python
from src.webapp.webhooks import webhook_manager, EventTypes

# Registrar un webhook para recibir notificaciones
webhook_id = webhook_manager.register_webhook(
    hospital_id=1,
    url="https://hospital-ejemplo.com/webhooks/salud-chilecito",
    events=[EventTypes.TURNO_CREATED, EventTypes.TURNO_CANCELLED],
    secret="secreto-para-firmar-payloads"
)
print(f"Webhook registrado: {webhook_id}")

# Disparar un evento de prueba
webhook_manager.trigger_event(
    EventTypes.TURNO_CREATED,
    {
        "id": 1,
        "paciente_id": 1,
        "medico_id": 1,
        "fecha": "2026-06-20",
        "hora": "09:30"
    }
)
```

### Ejemplo de uso de Adaptadores

```python
from src.webapp.adapters import AdapterFactory, EXAMPLE_REST_CONFIG

# Crear un adaptador REST para un HIS
config = EXAMPLE_REST_CONFIG.copy()
config["base_url"] = "https://api.hospital-ejemplo.com"
config["api_key"] = "tu-api-key-aqui"

adapter = AdapterFactory.create_adapter("rest", config)

# Sincronizar pacientes con el HIS
pacientes = [
    {"dni": "12345678", "nombre": "Juan Pérez", "telefono": "3825-123456"}
]
synced = adapter.sync_patients(pacientes)
print(f"Pacientes sincronizados: {len(synced)}")
```

### Arquitectura de Integración

```
Paciente → Salud Chilecito → API REST → Adaptador → HIS (Sistema existente)
         ←                ←         ←         ←
```

Salud Chilecito actúa como una **capa de mejora** que se conecta a los sistemas existentes de los hospitales, agregando funcionalidades como:
- Selección por síntomas con IA
- Bot conversacional
- Landing pages personalizadas
- Visualización mejorada de disponibilidad

### Documentación

- [Arquitectura de Integración](../docs/ARQUITECTURA_INTEGRACION.md) - Guía completa de integración
- [Documentación de API](../docs/API_OPENAPI.md) - Referencia completa de la API REST
- [Ejemplos de Integración](../examples/integracion_his.py) - Ejemplos de código completos

## 7. Resumen para el Profesor

**Modelo de negocio**: Sistema vendido a hospitales/clínicas (single-hospital instance)

**Autores**: Alesandro David Fajardo / Kevin Facundo Nunez  
**Universidad**: Universidad Nacional de Chilecito  
**Año**: 2026

### Características principales de la plataforma

1. **Gestión completa**: Centros, médicos, pacientes, turnos, agendas, documentos
2. **Selección por síntomas**: Derivación automática a especialidades (dolor de pecho → Cardiología)
3. **Precios personalizados**: Por especialidad y tipo de consulta (General, Urgencia, Seguimiento)
4. **Configuración por hospital**: Branding personalizado (logo, colores, mensajes de bienvenida)
5. **Disponibilidad en tiempo real**: Días y horarios específicos para los próximos 7 días
6. **Integración con HIS**: Se conecta a sistemas existentes, no los reemplaza
7. **Bot IA local**: Interfaz conversacional sin dependencias externas
8. **API REST estándar**: Para integración con otros sistemas

### Arquitectura técnica

- **Base de datos**: Oracle con scripts completos (tablespaces, usuarios, esquema, índices, seed)
- **Capa DAO**: Python con patrón Data Access Object para abstracción de base de datos
- **API REST**: Endpoints estándar con documentación OpenAPI
- **Frontend**: JavaScript con visualización mejorada de disponibilidad
- **Bot IA**: Local sin dependencias de servicios externos

### Ventajas competitivas

1. **No reemplaza**: Los hospitales mantienen sus sistemas existentes
2. **Fácil integración**: API estándar y documentación clara
3. **Valor agregado**: Funcionalidades que los HIS no tienen (IA, selección por síntomas)
4. **Flexibilidad**: Se adapta a diferentes sistemas mediante adaptadores
5. **Costo efectivo**: Menor costo que migrar a un sistema completo

### Comandos para ejecutar

```bash
# Instalar dependencias
python -m venv .venv
.venv\Scripts\activate        # Windows
source .venv/bin/activate      # Ubuntu
pip install -r requirements.txt

# Ejecutar servidor
python -m src.webapp.server
```

### URLs de acceso

- `http://localhost:8000` - Plataforma gráfica principal
- `http://localhost:8000/bot` - Bot IA conversacional
- `http://localhost:8000/landing` - Landing page del hospital

### Documentación completa

- [README](../README.md) - Documentación general del proyecto
- [Arquitectura de Integración](../docs/ARQUITECTURA_INTEGRACION.md) - Guía de integración con HIS
- [Documentación de API](../docs/API_OPENAPI.md) - Referencia completa de la API REST
- [Ejemplos de Integración](../examples/integracion_his.py) - Ejemplos de código completos
- [Integración con Hospital](../docs/INTEGRACION_HOSPITAL.md) - Guía de integración específica

## 7. Comandos finales de presentacion

```bash
python scripts/check_requirements.py
python -m pytest -q
python -m src.webapp.server
```

Abrir:

- `http://localhost:8000`
- `http://localhost:8000/bot`
